In [1]:

import requests

def explore_country():
    country = input("Enter a country name: ").strip()

    if not country:
        print("Please enter a country name.")
        return

    url = f"https://restcountries.com/v3.1/name/{country}"

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 404:
            print("Country not found. Please try again.")
            return

        response.raise_for_status()
        countries = response.json()
        data = countries[0]

        print("\n===== COUNTRY INFORMATION =====")
        print("Official name:", data["name"]["official"])
        print("Common name:", data["name"]["common"])
        print("Capital:", ", ".join(data.get("capital", ["Not available"])))
        print("Population:", f'{data["population"]:,}')

        area = data.get("area")
        print("Area:", f"{area:,.2f} km²" if area is not None else "Not available")

        currencies = data.get("currencies", {})
        if currencies:
            currency_list = [
                f'{code}: {details.get("name", "Unknown")} '
                f'({details.get("symbol", "No symbol")})'
                for code, details in currencies.items()
            ]
            print("Currencies:", ", ".join(currency_list))
        else:
            print("Currencies: Not available")

        languages = data.get("languages", {})
        print("Languages:", ", ".join(languages.values()) if languages else "Not available")

        flag = data.get("flag", "")
        print("Flag:", flag)

    except requests.exceptions.Timeout:
        print("The request timed out. Please try again.")
    except requests.exceptions.ConnectionError:
        print("Could not connect to the internet.")
    except requests.exceptions.RequestException as error:
        print("API error:", error)
    except (ValueError, KeyError, IndexError, TypeError):
        print("Could not read the country data correctly.")

if __name__ == "__main__":
    explore_country()


Could not read the country data correctly.


In [2]:

import json
from datetime import date, datetime
from pathlib import Path

DIARY_FILE = Path("diary.json")
EXPORT_FILE = Path("diary_export.txt")


def load_entries():
    try:
        if not DIARY_FILE.exists():
            return []

        with DIARY_FILE.open("r", encoding="utf-8") as file:
            entries = json.load(file)

        if not isinstance(entries, list):
            print("Diary file format is invalid. Starting a new diary.")
            return []

        return entries

    except (OSError, json.JSONDecodeError) as error:
        print("Could not read diary:", error)
        return []


def save_entries(entries):
    try:
        with DIARY_FILE.open("w", encoding="utf-8") as file:
            json.dump(entries, file, indent=4, ensure_ascii=False)
        return True

    except OSError as error:
        print("Could not save diary:", error)
        return False


def write_entry():
    entries = load_entries()

    title = input("Enter entry title: ").strip()
    content = input("Write your diary entry: ").strip()
    mood = input("Enter your mood: ").strip()

    if not title or not content or not mood:
        print("Title, content and mood cannot be empty.")
        return

    entry = {
        "date": date.today().isoformat(),
        "title": title,
        "content": content,
        "mood": mood
    }

    entries.append(entry)

    if save_entries(entries):
        print("Diary entry saved successfully!")


def read_today():
    entries = load_entries()
    today = date.today().isoformat()

    todays_entries = [
        entry for entry in entries
        if entry.get("date") == today
    ]

    if not todays_entries:
        print("You have no diary entries for today.")
        return

    for entry in todays_entries:
        print("\nDate:", entry.get("date", "Unknown"))
        print("Title:", entry.get("title", "Untitled"))
        print("Mood:", entry.get("mood", "Unknown"))
        print("Content:", entry.get("content", ""))


def search_entries():
    entries = load_entries()
    keyword = input("Enter a keyword to search: ").strip().lower()

    if not keyword:
        print("Please enter a keyword.")
        return

    matches = [
        entry for entry in entries
        if keyword in entry.get("title", "").lower()
        or keyword in entry.get("content", "").lower()
        or keyword in entry.get("mood", "").lower()
    ]

    if not matches:
        print("No matching diary entries found.")
        return

    for entry in matches:
        print("\nDate:", entry.get("date", "Unknown"))
        print("Title:", entry.get("title", "Untitled"))
        print("Mood:", entry.get("mood", "Unknown"))
        print("Content:", entry.get("content", ""))


def export_entries():
    entries = load_entries()

    try:
        with EXPORT_FILE.open("w", encoding="utf-8") as file:
            for entry in entries:
                file.write("=" * 35 + "\n")
                file.write(f'Date: {entry.get("date", "")}\n')
                file.write(f'Title: {entry.get("title", "")}\n')
                file.write(f'Mood: {entry.get("mood", "")}\n')
                file.write(f'Content: {entry.get("content", "")}\n\n')

        print("Diary exported to diary_export.txt")

    except OSError as error:
        print("Could not export diary:", error)


def main():
    while True:
        print("\n===== PERSONAL DIARY APP =====")
        print("1. Write a diary entry")
        print("2. Read today's entries")
        print("3. Search entries by keyword")
        print("4. Export diary to text file")
        print("5. Exit")

        choice = input("Choose an option (1-5): ").strip()

        if choice == "1":
            write_entry()
        elif choice == "2":
            read_today()
        elif choice == "3":
            search_entries()
        elif choice == "4":
            export_entries()
        elif choice == "5":
            print("Goodbye!")
            break
        else:
            print("Invalid choice. Enter a number from 1 to 5.")


if __name__ == "__main__":
    main()



===== PERSONAL DIARY APP =====
1. Write a diary entry
2. Read today's entries
3. Search entries by keyword
4. Export diary to text file
5. Exit
Goodbye!


In [ ]:

import json
import time
from datetime import datetime
from pathlib import Path

import requests

API_URL = "https://api.coingecko.com/api/v3/simple/price"
PRICE_FILE = Path("prices.json")

COINS = {
    "bitcoin": "BTC",
    "ethereum": "ETH",
    "dogecoin": "DOGE"
}

CURRENCIES = ["myr", "usd"]


def load_history():
    try:
        if not PRICE_FILE.exists():
            return []

        with PRICE_FILE.open("r", encoding="utf-8") as file:
            data = json.load(file)

        if isinstance(data, list):
            return data

        print("Price history has an invalid format. Starting a new history.")
        return []

    except (OSError, json.JSONDecodeError) as error:
        print("Could not load price history:", error)
        return []


def save_history(history):
    try:
        with PRICE_FILE.open("w", encoding="utf-8") as file:
            json.dump(history, file, indent=4)
        return True

    except OSError as error:
        print("Could not save prices:", error)
        return False


def fetch_prices():
    params = {
        "ids": ",".join(COINS.keys()),
        "vs_currencies": ",".join(CURRENCIES)
    }

    response = requests.get(API_URL, params=params, timeout=15)
    response.raise_for_status()
    return response.json()


def show_prices(prices, baseline):
    print("\n===== CRYPTO PRICE TRACKER =====")

    for coin_id, symbol in COINS.items():
        coin_data = prices.get(coin_id, {})

        print(f"\n{symbol} ({coin_id.title()})")

        for currency in CURRENCIES:
            current = coin_data.get(currency)

            if current is None:
                print(f"  {currency.upper()}: Price unavailable")
                continue

            print(f"  {currency.upper()}: {current:,.4f}")

            first = baseline.get(coin_id, {}).get(currency)

            if first is not None:
                change = current - first
                percent = (change / first * 100) if first != 0 else 0

                print(
                    f"  Change since first fetch: "
                    f"{change:+,.4f} ({percent:+.2f}%)"
                )
            else:
                print("  Change: Baseline recorded on this fetch")


def main():
    history = load_history()
    baseline = None

    # Recover the baseline from the earliest saved record.
    for record in history:
        if isinstance(record, dict) and isinstance(record.get("prices"), dict):
            baseline = record["prices"]
            break

    print("Crypto tracker started. Press Ctrl+C to stop.")

    try:
        while True:
            try:
                prices = fetch_prices()

                timestamp = datetime.now().astimezone().isoformat(
                    timespec="seconds"
                )

                record = {
                    "timestamp": timestamp,
                    "prices": prices
                }

                history.append(record)

                if save_history(history):
                    if baseline is None:
                        baseline = prices
                        print("\nFirst successful fetch recorded as baseline.")

                    print("\nTimestamp:", timestamp)
                    show_prices(prices, baseline)

                else:
                    history.pop()
                    print("This fetch could not be saved.")

            except requests.exceptions.RequestException as error:
                print("\nCould not fetch prices:", error)

            except (ValueError, TypeError) as error:
                print("\nCould not process price data:", error)

            print("\nNext update in 60 seconds.")
            time.sleep(60)

    except KeyboardInterrupt:
        print("\nTracker stopped. Your saved history is in prices.json.")


if __name__ == "__main__":
    main()


Crypto tracker started. Press Ctrl+C to stop.

First successful fetch recorded as baseline.

Timestamp: 2026-10-28T15:52:45+03:00

===== CRYPTO PRICE TRACKER =====

BTC (Bitcoin)
  MYR: 328,882.0000
  Change since first fetch: +0.0000 (+0.00%)
  USD: 80,676.0000
  Change since first fetch: +0.0000 (+0.00%)

ETH (Ethereum)
  MYR: 9,527.0300
  Change since first fetch: +0.0000 (+0.00%)
  USD: 2,337.0000
  Change since first fetch: +0.0000 (+0.00%)

DOGE (Dogecoin)
  MYR: 0.3229
  Change since first fetch: +0.0000 (+0.00%)
  USD: 0.0792
  Change since first fetch: +0.0000 (+0.00%)

Next update in 60 seconds.

Timestamp: 2026-10-28T15:53:46+03:00

===== CRYPTO PRICE TRACKER =====

BTC (Bitcoin)
  MYR: 328,937.0000
  Change since first fetch: +55.0000 (+0.02%)
  USD: 80,689.0000
  Change since first fetch: +13.0000 (+0.02%)

ETH (Ethereum)
  MYR: 9,529.9200
  Change since first fetch: +2.8900 (+0.03%)
  USD: 2,337.7100
  Change since first fetch: +0.7100 (+0.03%)

DOGE (Dogecoin)
  MYR: 0.3

In [ ]:

# Student Study Planner — Software Requirements Document (SRD)

## 1. App Name and Description

**App Name:** StudyMate

**Description:** StudyMate is an application that helps students organise their subjects, plan revision sessions, and track completed study tasks.

## 2. Problem Statement

Students often struggle to manage revision time, remember homework deadlines, and organise their subjects. StudyMate provides a simple place to plan study sessions and monitor progress.

## 3. Target User Profile

- Secondary school students aged 13–18.
- Students preparing for school examinations.
- Students who need help organising homework and revision.
- Users who prefer a simple and easy-to-use interface.

## 4. Three MVP Features

### Feature 1: Task Management
Users can add study tasks with a subject, description, and due date. They can mark tasks as completed.

### Feature 2: Study Schedule
Users can create and view planned study sessions, including the subject, date, and start time.

### Feature 3: Progress Tracking
Users can view the number of completed and remaining study tasks to monitor their revision progress.

## 5. Technology Stack and Justification

- **Python:** Main programming language because it is beginner-friendly and suitable for application logic.
- **JSON:** Stores study tasks and schedules in a simple file format.
- **Visual Studio Code:** Development environment for writing and testing the application.
- **Git:** Tracks code changes and helps recover previous versions.

This technology stack is appropriate for a small MVP because it is simple, affordable, and does not require a database server.

## 6. Data Model

Each study task contains the following fields:

- `id`: Unique identifier for the task.
- `subject`: School subject.
- `description`: Details of the study task.
- `due_date`: Task deadline in YYYY-MM-DD format.
- `completed`: Boolean value indicating whether the task is complete.

Example JSON record:

    {
        "id": 1,
        "subject": "Biology",
        "description": "Revise cell structure",
        "due_date": "2026-11-05",
        "completed": false
    }

Study tasks will be stored in a file named `tasks.json`.

## 7. Risks and Mitigations

### Risk 1: Data Loss
The application may lose data if the JSON file is accidentally deleted or corrupted.

**Mitigation:** Validate file contents, handle file errors, and provide a backup option.

### Risk 2: Missed Deadlines
Users may forget to check their planned study tasks.

**Mitigation:** Display upcoming deadlines clearly and highlight overdue tasks when the application is opened.

## 8. Success Metric

The MVP will be considered successful if at least 80% of 10 student testers can create a study task, schedule a revision session, and mark a task as completed without assistance during a one-week trial.
